<a href="https://colab.research.google.com/github/Abhi-pan1982/ML_NLP/blob/main/sentimental_class_model_build_class_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

##NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

##

#SKLEARN Libraries ffor sentiment classification

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier

##Evaluation Metrics

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

##Model Deployment

import pickle

##File upload library from google colab

from google.colab import files



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


LOAD Sentiment RAW DATA

In [ ]:
uploaded = files.upload()

csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename, encoding='latin1')
df.head()




Saving train.csv to train.csv


,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [ ]:
df.shape

(27481, 10)

In [ ]:
df.sentiment.value_counts()

,count
sentiment,
neutral,11118
positive,8582
negative,7781


Cleaning of Text

In [ ]:
def clean_text(text):
  #Convert all text to lower case
  text = str(text).lower()

  #Removal of Punctuations
  text = text.translate(str.maketrans('', '', string.punctuation))

  #Removal of any digit using regular expression
  text = re.sub(r'\d+', '', text)

  return text

In [ ]:
df['text'] = df['text'].astype(str)
df['cleaned_text'] = df['text'].apply(clean_text)
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²),cleaned_text
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60,id have responded if i were going
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105,sooo sad i will miss you here in san diego
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18,my boss is bullying me
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164,what interview leave me alone
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26,sons of why couldnt they put them on the rel...


In [ ]:
def get_wordnet_pos(pos_tag):
  if pos_tag.startswith('J'):
    return wordnet.ADJ
  elif pos_tag.startswith('V'):
    return wordnet.VERB
  elif pos_tag.startswith('N'):
    return wordnet.NOUN
  elif pos_tag.startswith('R'):
    return wordnet.ADV
  else:
    return wordnet.NOUN

In [ ]:
def preprocess_text(text):
  #Split Text in to Multiple tokens (word)
  words = word_tokenize(text)
  # Remove stop Word and apply Lamitize
  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()
  preporocessed_words = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_tag(word_tokenize(text)) if word not in stop_words]

  return ' '.join(preporocessed_words)
  #



In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Some common NLTK resources are downloaded above to prevent future LookupErrors.

In [ ]:
df['processed_text'] = df['cleaned_text'].apply(preprocess_text)
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²),cleaned_text,processed_text
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60,id have responded if i were going,id respond go
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105,sooo sad i will miss you here in san diego,sooo sad miss san diego
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18,my boss is bullying me,bos bully
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164,what interview leave me alone,interview leave alone
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26,sons of why couldnt they put them on the rel...,son couldnt put release already buy


Train Test Split

In [ ]:
X = df['processed_text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape
#

((21984,), (5497,), (21984,), (5497,))

In [ ]:
X[1:5]

,processed_text
1,sooo sad miss san diego
2,bos bully
3,interview leave alone
4,son couldnt put release already buy


TFIDF Vectorixzation

In [ ]:
tfidf = TfidfVectorizer()
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

### Save the fitted TF-IDF Vectorizer

To ensure consistent text preprocessing for new input, we need to save the `tfidf` vectorizer that was fitted on the training data. This will create the `tfidf_vectorizer.pkl` file, which can then be loaded by your FastAPI application.

In [ ]:
import pickle

with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(tfidf, file)

In [ ]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 153236 stored elements and shape (21984, 21483)>

Model Training----Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_vec, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
y_pred_rf = rf.predict(X_test_vec)

In [ ]:
y_pred_rf

array(['neutral', 'neutral', 'neutral', ..., 'negative', 'positive',
       'positive'], dtype=object)

In [ ]:
print('Accuracy Score ----', accuracy_score(y_test, y_pred_rf))
#

Accuracy Score ---- 0.7002001091504457


Save the best Model

In [ ]:
import pickle

with open('sentiment_model.pkl', 'wb') as file:
  pickle.dump(rf, file)

Load the Trained Model------The code snippet should be used where we deploy our tained model

In [ ]:
with open('sentiment_model.pkl', 'rb') as file:
  model = pickle.load(file)

Predict using the latest Loaded model

In [ ]:
test_text = ['This Movie is absolutely good']


In [ ]:
test_cleaned = [preprocess_text(clean_text(t)) for t in test_text]
test_vec = tfidf.transform(test_cleaned)

In [ ]:
print('Predicted Sentiment ------', model.predict(test_vec)[0])

Predicted Sentiment ------ positive


In [ ]:


from fastapi import FastAPI
from pydantic import BaseModel



# Initialize FastAPI app
app = FastAPI()

# Define request body model for input text
class TextInput(BaseModel):
    text: str

@app.post("/predict/")
async def predict_sentiment(item: TextInput):
    # Preprocess the input text
    cleaned_input = clean_text(item.text)
    processed_input = preprocess_text(cleaned_input)

    # Transform the text using the loaded TF-IDF vectorizer
    input_vec = tfidf_vectorizer.transform([processed_input])

    # Make prediction
    prediction = model.predict(input_vec)[0]

    return {"text": item.text, "predicted_sentiment": prediction}

# To run this script:
# 1. Save it as `app.py`.
# 2. Make sure `sentiment_model.pkl` and `tfidf_vectorizer.pkl` are in the same directory.
# 3. Install necessary packages: `pip install fastapi uvicorn nltk scikit-learn pandas`
# 4. Run from your terminal: `uvicorn app:app --reload`

In [ ]:
!uvicorn app:app --reload

INFO:     Will watch for changes in these directories: ['/content']
ERROR:    Error loading ASGI app. Could not import module "app".


## Containerizing the FastAPI App with Docker

To containerize your FastAPI application, you'll need two main files:

1.  **`requirements.txt`**: This file lists all the Python dependencies for your application.
2.  **`Dockerfile`**: This file contains the instructions for Docker to build your image.

### Step 1: Create `requirements.txt`

First, create a `requirements.txt` file in the same directory as your `app.py`. This file should contain all the Python packages your FastAPI application uses. You can generate this automatically if you're in a virtual environment by running `pip freeze > requirements.txt`.

For this project, your `requirements.txt` should look something like this:

```
fastapi
uvicorn
pydantic
nltk
scikit-learn
pandas
```

### Step 2: Create `Dockerfile`

Next, create a file named `Dockerfile` (no extension) in the same directory as your `app.py` and `requirements.txt`. Add the following content to it:

```dockerfile
# Use an official Python runtime as a parent image
FROM python:3.9-slim-buster

# Set the working directory in the container
WORKDIR /app

# Copy the requirements file into the container at /app
COPY requirements.txt .

# Install any needed packages specified in requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Download NLTK data (punkt, stopwords, wordnet, averaged_perceptron_tagger)
# This helps ensure NLTK data is available when the container starts without re-downloading.
# These can also be pre-downloaded in a build stage to reduce final image size if needed.
RUN python -c "import nltk; nltk.download('punkt'); nltk.download('averaged_perceptron_tagger'); nltk.download('stopwords'); nltk.download('wordnet');"

# Copy the application files into the container
COPY . .

# Expose the port that FastAPI will run on
EXPOSE 8000

# Define environment variable for FastAPI app (if your app.py is named differently, adjust here)
ENV APP_MODULE=app:app

# Run the FastAPI application using Uvicorn
CMD ["uvicorn", "--host", "0.0.0.0", "--port", "8000", "$APP_MODULE"]
```

### Step 3: Build the Docker Image

Open your terminal, navigate to the directory where your `Dockerfile`, `app.py`, `sentiment_model.pkl`, `tfidf_vectorizer.pkl`, and `requirements.txt` are located, and run the following command to build your Docker image:

```bash
docker build -t sentiment-api .
```

-   `-t sentiment-api`: Tags your image with the name `sentiment-api`. You can choose any name you like.
-   `.`: Indicates that the Dockerfile is in the current directory.

### Step 4: Run the Docker Container

Once the image is built, you can run your FastAPI application in a Docker container using the following command:

```bash
docker run -p 8000:8000 sentiment-api
```

-   `-p 8000:8000`: Maps port 8000 on your host machine to port 8000 inside the Docker container. This allows you to access your API from your browser or other clients.
-   `sentiment-api`: The name of the Docker image you just built.